# SATP Perpetrator Classification

This notebook implements multi-class text classification for identifying perpetrators in SATP incidents using transformer models.

## Notebook Structure

1. **Setup & Configuration** - Import libraries and configure models
2. **Data Management** - Load, clean, and split data  
3. **Model Components** - Define classes and functions
4. **Training Pipeline** - Run experiments across models and data fractions
5. **Results & Analysis** - Save results and generate visualizations

## Key Features
- Supports multiple transformer models (BERT, RoBERTa, DistilBERT, etc.)
- Tests performance across different data fractions (3% to 100%)
- Comprehensive evaluation with accuracy, F1-macro, and F1-weighted metrics
- Automated result saving and visualization generation

# 1. Setup & Configuration


## 1.1 Environment Setup

### 1.11. Colab Setup

In [ ]:
# Verify that master setup was run
import os
if not os.path.exists('/content/code-satp'):
    print("❌ ERROR: Repository not found!")
    print("🔧 SOLUTION: Please run the master setup notebook first:")
    print("   📁 /content/code-satp/colab-setup/00_master_setup.ipynb")
    raise FileNotFoundError("Run master setup first")

# Add cloned repository to Python path
import sys
sys.path.append('/content/code-satp/models/classification-models')

print("✅ Environment ready for classification experiments!")

### 1.12 Local Setup

In [ ]:
# # For local development, uncomment these lines:
# import sys
# sys.path.append('./utils')  # Add utils to path

# # Local data and results paths
# DATA_PATH = "../../data/"
# RESULTS_PATH = "./results/"

# # Create local results directory
# import os
# os.makedirs(RESULTS_PATH, exist_ok=True)

# # Verify GPU availability
# import torch
# if torch.cuda.is_available():
#     print(f"✅ GPU Available: {torch.cuda.get_device_name(0)}")
# else:
#     print("⚠️ GPU not available, using CPU")

# print("✅ Local setup complete!")

## 1.2 Import Libraries

In [ ]:
# Core libraries
import pandas as pd
import numpy as np
import json
import shutil
from pathlib import Path
from google.colab import drive

# Machine Learning
import torch
from torch.utils.data import Dataset
from torch.nn.functional import softmax
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

# Transformers
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

### Custom visualization modules
from visualization_utils import scatter_plot_speed_vs_accuracy, heatmap_label_f1_scores

# Set plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

## 1.3 GPU Configuration


In [ ]:
# Check for GPU availability
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"GPU is available. Using: {torch.cuda.get_device_name(0)}")
else:
    device = torch.device("cpu")
    print("GPU not available, using CPU.")

## 1.4 Experiment Configuration


In [ ]:
# Experiment configuration
EXPERIMENT_CONFIG = {
    # Data fractions to test model performance
    'fractions': [1/32, 1/16, 1/8, 1/4, 1/2, 1.0],
    
    # Model configurations (focusing on cased versions for better performance)
    'models': [
        "bert-base-cased",
        "snowood1/ConfliBERT-scr-cased",  # Specialized for conflict data
        "FacebookAI/roberta-base",
        "distilbert-base-cased",
        "xlnet-base-cased", 
        "google/electra-base-discriminator"
    ],
    
    # Training parameters
    'epochs': 2,
    'batch_size': 32,
    'max_length': 128
}

# Human-readable labels for models and fractions
fraction_labels = {
    1/32: "3%", 1/16: "6%", 1/8: "12%", 
    1/4: "25%", 1/2: "50%", 1.0: "100%"
}

model_name_labels = {
    "bert-base-cased": "BERT",
    "snowood1/ConfliBERT-scr-cased": "ConfliBERT", 
    "FacebookAI/roberta-base": "RoBERTa",
    "distilbert-base-cased": "DistilBERT",
    "xlnet-base-cased": "XLNet",
    "google/electra-base-discriminator": "ELECTRA"
}

print("🔧 Experiment Configuration:")
print(f"📊 Data fractions: {len(EXPERIMENT_CONFIG['fractions'])} ({list(fraction_labels.values())})")
print(f"🤖 Models: {len(EXPERIMENT_CONFIG['models'])} transformer models")
print(f"⚙️  Training: {EXPERIMENT_CONFIG['epochs']} epochs, batch size {EXPERIMENT_CONFIG['batch_size']}")
print(f"📈 Total experiments: {len(EXPERIMENT_CONFIG['fractions']) * len(EXPERIMENT_CONFIG['models'])}")


# 2. Data Management

## 2.1 Load Data

In [ ]:
# Read data from cloned repository
try:
    data = pd.read_csv('/content/code-satp/data/perpetrator.csv')
    print("✅ Data loaded successfully from cloned repository")
    print(data.head())
except Exception as e:
    print(f"❌ Error loading CSV from cloned repo: {e}")
    print("🔧 Fallback: Trying GitHub URL...")

# Fallback to GitHub if cloned repo fails or if working locally
url = 'https://raw.githubusercontent.com/eteitelbaum/code-satp/main/data/perpetrator.csv'
try:
    data = pd.read_csv(url)
    print("✅ Data loaded successfully from GitHub")
    print(data.head())
except Exception as e:
    print(f"❌ Error loading CSV from GitHub: {e}")

## 2.2 Data Exploration & Cleaning


In [ ]:
# Explore and clean the data
print("📊 Dataset Information:")
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"\nUnique perpetrator values: {df['perpetrator'].unique()}")
print(f"\nPerpetrator distribution:")
print(df['perpetrator'].value_counts())
print(f"\nMissing values:")
print(df.isnull().sum())

# Data cleaning steps
print("\n🧹 Data cleaning steps:")

# 1. Select necessary columns
df_clean = df[['incident_summary', 'perpetrator']].copy()
print(f"1. Selected columns: {list(df_clean.columns)}")

# 2. Remove missing values
initial_shape = df_clean.shape[0]
df_clean.dropna(inplace=True)
final_shape = df_clean.shape[0]
print(f"2. Removed {initial_shape - final_shape} rows with missing values")

# 3. Remove potential duplicates
initial_shape = df_clean.shape[0]
df_clean.drop_duplicates(inplace=True)
final_shape = df_clean.shape[0]
print(f"3. Removed {initial_shape - final_shape} duplicate rows")

# 4. Clean text
df_clean['incident_summary'] = df_clean['incident_summary'].str.strip()
print("4. Cleaned whitespace from text")

print(f"\n✅ Final cleaned dataset shape: {df_clean.shape}")
print(f"Final perpetrator distribution:")
print(df_clean['perpetrator'].value_counts())


## 2.3 Train-Test Split


In [ ]:
# Create stratified train-validation-test splits
print("🔄 Creating train-validation-test splits...")

# First split: separate training data from temp (test + validation)
train_df, temp_df = train_test_split(
    df_clean, 
    test_size=0.2, 
    stratify=df_clean['perpetrator'], 
    random_state=42
)

# Second split: divide temp into validation and test
val_df, test_df = train_test_split(
    temp_df, 
    test_size=0.5, 
    stratify=temp_df['perpetrator'], 
    random_state=42
)

print(f"Training set shape: {train_df.shape}")
print(f"Validation set shape: {val_df.shape}")
print(f"Test set shape: {test_df.shape}")

# Verify stratification
print("\n📊 Label Distribution Verification:")
print("\nTraining set distribution:")
train_dist = train_df['perpetrator'].value_counts(normalize=True).sort_index()
print(train_dist)

print("\nValidation set distribution:")
val_dist = val_df['perpetrator'].value_counts(normalize=True).sort_index()
print(val_dist)

print("\nTest set distribution:")
test_dist = test_df['perpetrator'].value_counts(normalize=True).sort_index()
print(test_dist)

print("\n📈 Absolute Counts:")
print("Training:", train_df['perpetrator'].value_counts().sort_index())
print("Validation:", val_df['perpetrator'].value_counts().sort_index())
print("Test:", test_df['perpetrator'].value_counts().sort_index())


# 3. Model Components & Training

## 3.1 Dataset Class & Training Functions

### 3.11 Single Label Dataset class

In [ ]:
class SingleLabelDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        """
        Args:
            texts: List of text strings.
            labels: List of integer class labels (e.g., [0, 1, 2, ...]).
            tokenizer: A Hugging Face AutoTokenizer.
            max_length: Max sequence length for tokenization.
        """
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]

        encoding = self.tokenizer(
            text,
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        # Squeeze batch dimension
        item = {k: v.squeeze() for k, v in encoding.items()}
        # Integer labels for cross-entropy
        item["labels"] = torch.tensor(label, dtype=torch.long)

        return item

### 3.12 Compute Metrics Function

In [ ]:
def compute_metrics(eval_pred, label_names=None):
    """
    Returns the entire classification_report (per-class metrics),
    as well as an 'accuracy' top-level key for easy access.
    """
    logits, labels = eval_pred
    preds = torch.argmax(torch.tensor(logits), dim=1).numpy()

    # Accuracy
    acc = accuracy_score(labels, preds)

    # Full classification report (dict)
    # This includes one entry per label/class, plus
    #   "accuracy", "macro avg", and "weighted avg".
    report_dict = classification_report(
        labels,
        preds,
        target_names=label_names if label_names else None,
        zero_division=0,
        output_dict=True
    )

    # Print a readable version, if desired:
    if label_names:
        print("\nFull Classification Report:\n",
              classification_report(labels, preds, target_names=label_names, zero_division=0))
    else:
        print("\nFull Classification Report:\n",
              classification_report(labels, preds, zero_division=0))

    # Return a dict that:
    #   - includes a top-level 'accuracy'
    #   - stores the entire classification_report in 'class_report'
    return {
        "accuracy": acc,
        "f1_macro": report_dict["macro avg"]["f1-score"],
        "f1_weighted": report_dict["weighted avg"]["f1-score"],
        "class_report": json.dumps(report_dict)  # keep full report in JSON-safe string explicitly
    }

### 3.13 Train Multiclass Model Function

In [ ]:
def train_multiclass_model(
    train_df,
    test_df,
    val_df,
    text_col="incident_summary",
    label_col="perpetrator",
    model_name="bert-base-uncased",
    epochs=2,
    batch_size=8
):
    """
    Trains a multi-class classifier with separate train, val, and test sets.

    Args:
        data_path (str): CSV file path. Must contain `text_col` and `label_col`.
        text_col (str): Name of the column containing the text.
        label_col (str): Name of the column containing the class label (string form).
        model_name (str): HF model identifier, e.g. 'bert-base-uncased', 'roberta-base', etc.
        test_size (float): Fraction of entire dataset to hold out for final test set.
        val_size (float): Fraction of entire dataset to hold out for validation set
                          (relative to total, not just leftover).
        epochs (int): Number of training epochs.
        batch_size (int): Batch size for training and evaluation.

    Returns:
        model: The trained model (same as trainer.model).
        tokenizer: The tokenizer used.
        test_metrics: Evaluation metrics on the test set.
    """


    # Convert label from string to integer IDs
    unique_labels = train_df[label_col].unique()
    label2id = {label: i for i, label in enumerate(unique_labels)}
    id2label = {v: k for k, v in label2id.items()}

    train_df["label_id"] = train_df[label_col].map(label2id)
    val_df["label_id"] = val_df[label_col].map(label2id)
    test_df["label_id"] = test_df[label_col].map(label2id)


    # Prepare data lists
    train_texts = train_df[text_col].tolist()
    train_labels = train_df["label_id"].tolist()

    val_texts = val_df[text_col].tolist()
    val_labels = val_df["label_id"].tolist()

    test_texts = test_df[text_col].tolist()
    test_labels = test_df["label_id"].tolist()

    # Tokenizer & Model
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    num_labels = len(unique_labels)

    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=num_labels
    )
    model.to(device)


    #  Create Dataset objects
    train_dataset = SingleLabelDataset(train_texts, train_labels, tokenizer)
    val_dataset = SingleLabelDataset(val_texts, val_labels, tokenizer)
    test_dataset = SingleLabelDataset(test_texts, test_labels, tokenizer)

    # Training Arguments
    training_args = TrainingArguments(
        output_dir="./model_output",
        num_train_epochs=epochs,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        eval_strategy="epoch",
        save_strategy="epoch",
        logging_dir="./logs",
        logging_steps=10,
        report_to="none",
        load_best_model_at_end=False  # set to True if you'd like to restore best checkpoint
    )

    # Create Trainer
    label_names = [id2label[i] for i in range(num_labels)]

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        compute_metrics=lambda eval_pred: compute_metrics(eval_pred, label_names=label_names)
    )

    # Train
    trainer.train()

    # Evaluate on the test set
    #     By default, trainer.evaluate() returns a dictionary of metric values
    test_metrics = trainer.evaluate(test_dataset)

    print("\nTest Set Evaluation Results:", test_metrics)
    # Evaluate on test set
    test_metrics = trainer.evaluate(test_dataset)
    print("\nTest Set Evaluation Results:", test_metrics)

    # === PREDICTIONS on TEST ===
    # Use trainer.predict instead of evaluate
    predictions_output = trainer.predict(test_dataset)
    logits = predictions_output.predictions
    probs = softmax(torch.tensor(logits), dim=1).numpy()
    preds = np.argmax(logits, axis=1)

    pred_df = pd.DataFrame({
        "incident_summary": test_df[text_col].tolist(),
        "true_label_id": test_labels,
        "true_label": [id2label[i] for i in test_labels],
        "pred_label_id": preds.tolist(),
        "pred_label": [id2label[i] for i in preds],
        "logits": logits.tolist(),
        "probabilities": probs.tolist()
    })

    return trainer.model, tokenizer, test_metrics, pred_df, id2label

### 3.14 Run Experiments Function

In [ ]:
def run_all_experiments_and_save(df_full, output_csv="experiment_results.csv", predictions_csv="predictions.csv"):
    """
    1. Iterates over fractions & model list
    2. Samples df_full by 'frac'
    3. Trains & evaluates using train_multiclass_model
    4. Saves results in a DataFrame, including:
       - Full classification report (flattened)
       - Training label-wise support
    5. Exports to CSV
    """
    results_list = []
    all_predictions = []

    for frac in fractions:
        subset_size = int(len(df_full) * frac)
        df_subset = df_full.sample(n=subset_size, random_state=42)

        frac_label = fraction_labels.get(frac, f"{frac*100:.1f}%")
        print(f"\n=== DATA FRACTION: {frac} ({subset_size} rows) ===")

        for model_name in models_list:
            model_label = model_name_labels.get(model_name, model_name)
            print(f"Training model: {model_label}")


            model, tokenizer, test_metrics, pred_df, id2label = train_multiclass_model(
                df_subset,
                test_df,
                val_df,
                text_col="incident_summary",
                label_col="perpetrator",
                model_name=model_name,
                epochs=2,
                batch_size=32
            )

            run_result = {
                "fraction_raw": frac,
                "fraction_label": frac_label,
                "subset_size": subset_size,
                "model_raw": model_name,
                "model_label": model_label
            }


            for key, val in test_metrics.items():
                if key == "class_report":
                    for label_name, metrics_dict in val.items():
                        if isinstance(metrics_dict, dict):
                            for metric_name, metric_value in metrics_dict.items():
                                safe_label = label_name.replace(" ", "_")
                                run_result[f"{safe_label}_{metric_name}"] = metric_value
                        else:
                            run_result[label_name] = metrics_dict
                else:
                    run_result[key] = val



            label_counts = df_subset["label_id"].value_counts()
            for label_id_val, count in label_counts.items():
                label_name = id2label[label_id_val]
                run_result[f"train_support_{label_name}"] = count

            results_list.append(run_result)

            pred_df["model"] = model_name
            pred_df["model_label"] = model_label
            pred_df["fraction"] = frac
            pred_df["fraction_label"] = frac_label
            all_predictions.append(pred_df)

    results_df = pd.DataFrame(results_list)
    results_df.to_csv(output_csv, index=False)
    full_pred_df = pd.concat(all_predictions, ignore_index=True)
    full_pred_df.to_csv(predictions_csv, index=False)

    print(f"Test results saved to {output_csv}")
    print(f"Test predictions saved to {predictions_csv}")
    return results_df, full_pred_df

## 3.2 Training Pipeline

### 3.21 Run Training Loop

In [ ]:
# Run models
final_results_df, test_predictions_df  = run_all_experiments_and_save(train_df, output_csv="experiment_results.csv", predictions_csv="predictions.csv" )

# Inspect final_results_df in Python:
print(final_results_df.head())

### 3.22 Save Results

#### Save to JSON or CSV

In [ ]:
#final_results_df.to_json("/content/drive/MyDrive/SATP_data/paper/exp/perpetrator/experiment_results.json", orient="records")
#final_results_df.to_csv("/content/drive/MyDrive/SATP_data/paper/exp/perpetrator/experiment_results.csv", index=False)
final_results_df.to_csv("/content/drive/MyDrive/colab/satp-results/perpetrator/perpetrator_summary.csv", index=False)
test_predictions_df.to_csv("/content/drive/MyDrive/colab/satp-results/perpetrator/perpetrator_predictions.csv", index=False)

#### Check That Results are Saved

In [ ]:
!ls "/content/drive/MyDrive/colab/satp-results/perpetrator"

## 3.3 Results & Analysis

### 3.31 Import Results

In [ ]:
#df = pd.read_csv('/content/drive/MyDrive/SATP_data/paper/exp/perpetrator/experiment_results.csv')
df = pd.read_csv("/content/drive/MyDrive/colab/satp-results/perpetrator_summary.csv")

#### Check the Data

In [ ]:
df.columns

### 3.32 Visualization

#### Line Chart of Accuracy vs. Fraction of Data

In [ ]:
plt.figure(figsize=(10, 6))
sns.lineplot(data=final_results_df, x="fraction_label", y="eval_accuracy", hue="model_label", marker="o")

plt.xlabel("Fraction of Data")
plt.ylabel("Evaluation Accuracy") 
plt.title("Evaluation Accuracy vs. Fraction of Data for Different Models")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

#### Scatter Plot of Accuracy vs. Speed

In [ ]:
# Using the visualization utility function for scatter plot
scatter_plot_speed_vs_accuracy(
    df=final_results_df,
    x_col="eval_samples_per_second",
    y_col="eval_accuracy", 
    hue_col="model_label",
    size_col="fraction_raw",
    title="Accuracy vs. Throughput (Samples/Sec)"
)

#### Heatmaps of F1 Scores

In [ ]:
# Generate accuracy heatmap
plot_heatmap(final_results_df, "eval_accuracy")

# Generate F1-score heatmaps
plot_heatmap(final_results_df, "eval_f1_macro", " (Macro F1)")
plot_heatmap(final_results_df, "eval_f1_weighted", " (Weighted F1)")
